### Dinámica Molecular de Gas Hidrógeno ($H_2$)

En esta simulación, modelamos un ensamble de $N = 125$ moléculas de gas hidrógeno confinadas en una caja tridimensional de dimensiones $L_x = L_y = L_z = L$.

**Ecuaciones de Movimiento e Integración:**
Dado que asumimos un comportamiento de gas ideal (interacciones moleculares despreciables), las partículas viajan en línea recta hasta colisionar con una pared. Para la integración numérica utilizamos el método de Euler explícito:
$$r_i(t + \Delta t) = r_i(t) + v_i(t)\Delta t$$

**Colisiones con las paredes:**
Se implementan condiciones de frontera reflectantes. Si una partícula choca contra la pared en $x=0$ o $x=L$, su velocidad en ese eje se invierte ($v_{i,x} \to -v_{i,x}$).

**Propiedades Termodinámicas:**
1. **Temperatura:** Calculada a partir del Teorema de Equipartición de la Energía.
   $$T = \frac{2}{3} \frac{E_k}{N k_B}$$
2. **Presión:** Calculada mediante el cambio de momento lineal de las partículas al chocar contra las paredes ($\Delta p = 2m|v_\perp|$).
   $$P = \frac{1}{A} \frac{\Delta p}{\Delta t}$$
3. **Distribución de Maxwell-Boltzmann:**
   $$f(v) = 4\pi v^2 \left( \frac{m}{2\pi k_B T} \right)^{3/2} \exp\left( - \frac{m v^2}{2 k_B T} \right)$$
   

In [1]:
import numpy as np
import matplotlib.pyplot as plt
from ipywidgets import interact, FloatSlider, Layout
import IPython.display as display

m = 3.32e-27      # Masa de H2 (kg)
kB = 1.380649e-23 # Constante de Boltzmann (J/K)
dt = 1e-12        # Paso de tiempo: 1 ps
N_part = 125      # Número de moléculas
pasos = 1000      # Pasos de tiempo a simular por cada renderizado

def simular_gas_interactivo(T_target, L_micras):
    # Convertimos la longitud de micrómetros a metros
    L = L_micras * 1e-6 
    volumen = L**3
    area_total = 6 * (L**2)
    
    # Inicialización de posiciones (aleatorias sin salirse de la caja)
    pos = np.random.uniform(0, L, (N_part, 3))
    
    # Inicialización de velocidades (basado en T_target)
    # La desviación estándar de la velocidad en 1D es sqrt(kB*T / m)
    v_std = np.sqrt(kB * T_target / m)
    v = np.random.normal(0, v_std, (N_part, 3))
    
    # Arreglos para guardar la evolución de los últimos 1000 pasos
    T_hist = np.zeros(pasos)
    P_hist = np.zeros(pasos)
    Ek_hist = np.zeros(pasos)
    
    # 3. Integración del movimiento y colisiones
    for step in range(pasos):
        # 3.1 Mover partículas (Método de Euler)
        pos += v * dt
        
        dp_paredes = 0.0 # Acumulador de momento transferido a las paredes
        
        # 3.2 Colisiones elásticas con las paredes en los ejes X, Y, Z
        for i in range(3): 
            # Pared inferior (0)
            mask_inf = pos[:, i] < 0
            pos[mask_inf, i] *= -1
            v[mask_inf, i] *= -1
            dp_paredes += np.sum(2 * m * np.abs(v[mask_inf, i]))
            
            # Pared superior (L)
            mask_sup = pos[:, i] > L
            pos[mask_sup, i] = 2*L - pos[mask_sup, i] # Rebote exacto
            v[mask_sup, i] *= -1
            dp_paredes += np.sum(2 * m * np.abs(v[mask_sup, i]))
            
        # 3.3 Calcular propiedades macroscópicas
        # Energía Cinética Total (Suma de 1/2 m v^2 de todas las partículas)
        Ek_total = np.sum(0.5 * m * np.linalg.norm(v, axis=1)**2)
        
        # Temperatura (T = 2*Ek / 3*N*kB)
        T_inst = (2 * Ek_total) / (3 * N_part * kB)
        
        # Presión instantánea (F/A = (dp/dt)/Area)
        P_inst = (dp_paredes / dt) / area_total
        
        # Guardamos en el historial
        Ek_hist[step] = Ek_total
        T_hist[step] = T_inst
        P_hist[step] = P_inst


    fig, axs = plt.subplots(2, 2, figsize=(14, 10))
    tiempo_eje = np.arange(pasos) * dt * 1e12 # Eje de tiempo en picosegundos
    
    # Gráfico 1: Evolución de la Temperatura
    axs[0, 0].plot(tiempo_eje, T_hist, color='tomato', alpha=0.8)
    axs[0, 0].axhline(y=T_target, color='r', linestyle='--', label=f'T Objetivo ({T_target} K)')
    axs[0, 0].set_title('Evolución de la Temperatura')
    axs[0, 0].set_xlabel('Tiempo (ps)')
    axs[0, 0].set_ylabel('Temperatura (K)')
    axs[0, 0].legend()
    axs[0, 0].grid(True, alpha=0.3)
    
    # Gráfico 2: Evolución de la Presión (Ruido típico de N pequeño)
    axs[0, 1].plot(tiempo_eje, P_hist, color='steelblue', alpha=0.6)
    # Línea teórica: P = N*kB*T / V
    P_teorica = (N_part * kB * T_target) / volumen
    axs[0, 1].axhline(y=P_teorica, color='b', linestyle='--', label=f'P Ideal ({P_teorica:.1e} Pa)')
    axs[0, 1].set_title('Evolución de la Presión')
    axs[0, 1].set_xlabel('Tiempo (ps)')
    axs[0, 1].set_ylabel('Presión (Pa)')
    axs[0, 1].legend()
    axs[0, 1].grid(True, alpha=0.3)
    
    # Gráfico 3: Evolución de la Energía Cinética
    axs[1, 0].plot(tiempo_eje, Ek_hist, color='forestgreen', alpha=0.8)
    axs[1, 0].set_title('Evolución de la Energía Cinética')
    axs[1, 0].set_xlabel('Tiempo (ps)')
    axs[1, 0].set_ylabel('Energía Cinética (J)')
    axs[1, 0].grid(True, alpha=0.3)
    
    # Gráfico 4: Distribución de Velocidades vs Maxwell-Boltzmann
    v_mag = np.linalg.norm(v, axis=1) # Magnitud de la velocidad final
    axs[1, 1].hist(v_mag, bins=15, density=True, alpha=0.6, color='purple', label='Simulación')
    
    # Curva Teórica Maxwell-Boltzmann 3D
    v_teo = np.linspace(0, np.max(v_mag)*1.5, 100)
    f_v = 4 * np.pi * (v_teo**2) * ((m / (2 * np.pi * kB * T_target))**(1.5)) * np.exp((-m * v_teo**2) / (2 * kB * T_target))
    axs[1, 1].plot(v_teo, f_v, 'k-', lw=2, label='Teórica (Maxwell-Boltzmann)')
    
    axs[1, 1].set_title('Distribución de Velocidades')
    axs[1, 1].set_xlabel('Velocidad (m/s)')
    axs[1, 1].set_ylabel('Densidad de Probabilidad')
    axs[1, 1].legend()
    axs[1, 1].grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()


# Tarea 5: Controles deslizantes para Temperatura y Tamaño de la caja
print("Deslizadores para recalcular la simulación en tiempo real:")
interact(simular_gas_interactivo, 
         T_target=FloatSlider(value=300.0, min=50.0, max=1000.0, step=50.0, description='T_0 (K):', layout=Layout(width='500px')),
         L_micras=FloatSlider(value=10.0, min=1.0, max=50.0, step=1.0, description='Lado L (µm):', layout=Layout(width='500px')))

Deslizadores para recalcular la simulación en tiempo real:


interactive(children=(FloatSlider(value=300.0, description='T_0 (K):', layout=Layout(width='500px'), max=1000.…

<function __main__.simular_gas_interactivo(T_target, L_micras)>